In [1]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from scipy.linalg import expm

### Luokkien määrittely

In [145]:
class Transmon:

    def __init__(self,E_C,E_J,N):

        self.E_C=E_C #Varausenergia
        self.E_J=E_J #Josephsonin energia
        self.N=N   

        self.phi = np.linspace(-np.pi, np.pi, N, endpoint=False) #vaiheoperaattori diskretisoituna välillä [-pi,pi] 1000 pisteeseen
        self.delta_phi = self.phi[2]-self.phi[1] #vaiheoperaattorin diskresitoitu askelväli

        M = np.eye(N,k=1)+np.eye(N,k=-1)-2*np.eye(N) #Muodostetaan numero-operaattorin neliön matriisiesitys differenssimenetelmällä
        M[0,-1]=1
        M[-1,0]=1
        n_squared = -M/(self.delta_phi**2)

        self.H_0 = 4*E_C*n_squared-E_J*np.diag(np.cos(self.phi)) #Hamiltonin operaattori

        energies, eigenstates = np.linalg.eigh(self.H_0) # Transmonin ominaisenergiat ja -tilat

        self.energies = energies #Ominaisenergiat
        self.eigenstates = eigenstates #Ominaistilat
        self.frequency = energies[1]-energies[0] #Kubitin taajuus
    
    def H_D_eigbasis(self,dim): #Ajettu Hamiltonin operaattori ominaiskannassa
 
        H_0 = np.diag(self.energies[0:dim]-self.energies[0]) #Diagonaalimatriisi ominaisenergioista (lukumäärä ensimmäiset dim)

        M = np.zeros((dim, dim))
        vals = np.sqrt(np.arange(1, dim))
        M[np.arange(dim-1), np.arange(1, dim)] = vals      
        M[np.arange(1, dim), np.arange(dim-1)] = vals

        def H_t(A): return H_0 + A*M

        return H_t
    
class time_evolution:

    def __init__(self,generator):
        self.generator = generator
    
    def U(self, A, dt): return expm(-1j * self.generator(A) * dt) #Aikaevoluutio-operaattori

    def U_floquet(self, A_vals, dt): #Floquet-operaattori

        U_f = self.U(0, 0) #Aikakehitysoperaattori yhdelle Floquet-jaksolle
        for A in A_vals: 
            U_f = self.U(A, dt) @ U_f
        return U_f

class Pulse:

    def __init__(self,frequency,f_supp,pulse_funcs,pulse_times,time_unit=1):

        #Tarkistetaan, että on annettu sama määrä pulssifunktiota ja -aikoja
        if len(pulse_times)!=len(pulse_funcs):
            raise Exception("Pulse function and time interval arrays must have the same size") 

        #Määritellään ajan arvot eri pulssin osille
        piecewise_time_vals=[(np.arange(time)*time_unit) for time in pulse_times]

        #Ajan arvot koko pulssin ajalle
        self.time_vals = np.arange(np.sum(pulse_times))*time_unit

        #Pulssin amplitudi määriteltynä osissa
        self.piecewise_envelope_vals=[np.array([pulse_funcs[i](t) for t in piecewise_time_vals[i]]) for i in range(0,len(piecewise_time_vals))]

        #Pulssin amplitudi
        self.envelope = np.concatenate(self.piecewise_envelope_vals)

        #Derivaatta amplitudifunktiosta
        self.envelope_derivative = np.gradient(self.envelope,time_unit)
        
        #Määritellään signaalin vaihe siten, että amplitudi on nolla toisen jakson alussa
        #offset=frequency*pulse_times[0]
        offset=0

        #Aikapisteiden määrä
        N=len(self.time_vals)

        signal=np.zeros(N)
        cosine_signal=np.zeros(N)
        self.drag_component=np.zeros(N)

        #Määritellään signaali sini- ja kosinifunktioiden avulla, käyttäen taajuutta amplitudin funktiona
        signal = np.sin(frequency*self.time_vals-offset)
        cosine_signal = np.cos(frequency*self.time_vals-offset)

        #DRAG-komponentti pulssille
        self.drag_component = self.envelope_derivative * cosine_signal / (f_supp-frequency)

        #Signaalin arvot moduloituna amplitudilla
        self.raw = self.envelope*signal        

        self.freqs = np.fft.rfftfreq(len(self.time_vals), d=time_unit) #Taajuusavaruus diskretoituna time_unit välein
        self.fourier = np.fft.rfft(self.raw) #Fourier-muunnos signaalista

        #Määritellään DRAG-pulssi, ja sen fouriermuunnos
        self.drag_pulse = self.raw + self.drag_component
        self.drag_fourier = np.fft.rfft(self.drag_pulse)

### Funktioita pulssimuotojen muodostamiseen

In [85]:
def zero(t): #Nollafunktio
    return 0

def one(t): #Ykkösfunktio
    return 1

def linear(k,t_0,c): #Suora
    return lambda t: k*(t-t_0)+c

def linear_opp(k,t_0,c): #Suora -t-funktiona
    return lambda t: k*(t_0-t)+c

def arctan(t_0,c): #Arkustangentti skaalattuna välille [-1,1], siirrettynä t_0 verran ajassa
    return lambda t: np.arctan(c*(t-t_0))/np.pi+0.5

def arctan_opp(t_0,c): #Arkustangentti -t-funktiona, siirrettynä -t_0 verran ajassa. Pulssin laskevaa reunaa varten
    return lambda t: np.arctan(c*(t_0-t))/np.pi+0.5

def gaussian(t_0,sigma): #Normalisoitu Gaussinen funktio keskihajonnalla sigma, ja keskipisteellä t_0.
    return lambda t: np.exp(-((t-t_0)**2)/(2*sigma**2))
    
def gaussian_opp(t_0,sigma): #Normalisoitu Gaussinen funktio keskihajonnalla sigma, ja keskipisteellä -t_0. Pulssin laskevaa reunaa varten
    return lambda t: np.exp(-((t+t_0)**2)/(2*sigma**2))

### Transmonin määrittely

In [86]:
E_C = 0.3 #Varausenergia (GHz)
E_J = 8 #Josephsonin energia (GHz)

qubit=Transmon(E_C,E_J,500)

In [87]:
print("Transmonin yksitoista ensimmäistä energiatasoa (GHz), kun E_0=0")
print(qubit.energies[0:11]-qubit.energies[0])

Transmonin yksitoista ensimmäistä energiatasoa (GHz), kun E_0=0
[ 0.          4.0566369   7.73020385 11.07420845 13.12309129 17.41456479
 17.57786409 25.51228464 25.51356975 36.14800094 36.14800387]


## Simulaatio

In [157]:
print(qubit.frequency)

4.056636903636482


In [ ]:
f_ef = qubit.energies[2]-qubit.energies[1] #ef-siirtymän taajuus


In [181]:
dim = 7

evolution_eigenbasis=time_evolution(qubit.H_D_eigbasis(dim))

### Simulaatio alkuperäiselle pulssille

In [ ]:
def sim_pulse(f_d,A,evolution,dim,f_rabi):
    T_floquet = 2*np.pi/(f_d)
    dt = T_floquet/20

    N_edge=int(10/dt)
    N_pulse=int(1/(f_rabi*dt))

    pulse = Pulse(f_d,f_ef,[gaussian(10,2.5),one,gaussian_opp(0,2.5)],[N_edge,N_pulse,N_edge],time_unit=dt)
    U_floquet=evolution.U_floquet(A*pulse.raw[N_edge+100:N_edge+120],dt)

    psi_eig_accum = np.zeros(dim)
    psi_eig_accum[0] = 1
    
    last_vals=[psi_eig_accum]

    for i in range(0, N_edge): #Aikakehitys
        psi_eig_accum = evolution_eigenbasis.U(A*pulse.raw[i], dt) @ psi_eig_accum

    N_sim=int(N_pulse/20)+1

    for k in range(1,N_sim):
        psi_eig_accum = U_floquet @ psi_eig_accum
        psi_eig=psi_eig_accum

        pulse=Pulse(f_d,f_ef,[gaussian(10,2.5),one,gaussian_opp(0,2.5)],[N_edge,20*k,N_edge],time_unit=dt)
        for i in range(1, N_edge): #Aikakehitys
            psi_eig = evolution_eigenbasis.U(A*pulse.raw[N_edge+20*k+i], dt) @ psi_eig
        last_vals.append(np.abs(psi_eig)**2)

    e_vals = np.array([a[1] for a in last_vals])
    excited_max=max(e_vals)
    T_pulse = np.argmax(e_vals)*20*dt
    pulse_times=np.arange(0,N_sim)*20*dt


    return (excited_max,T_pulse,last_vals,pulse_times)

def find_params(q,A,evolution,f_rabi):
    min_range = -1000
    max_range = -200

    f_d = q.frequency/3 #Ajotaajuus
    pulse_times=[]
    max_e_vals=[]

    for i in range(min_range,max_range):
        sim_vals = sim_pulse(f_d+i/1000,A,evolution,dim,f_rabi)
        AC_stark_vals.append(AC_stark+i/1000)
        max_e_vals.append(sim_vals[0])
        pulse_times.append(sim_vals[1])

    plt.figure()
    plt.plot(AC_stark_vals, max_e_vals)
    plt.show()

    index=np.argmax(max_e_vals)
    e_max=max_e_vals[index]
    pi_pulse=AC_stark_vals[index]

    print(e_max)
    print(pi_pulse)
    
    return(e_max,pi_pulse)

In [ ]:
deltas = [] #AC Stark siirtymät
pulse_lengths = [] #Rabi-taajuudet
max_vals = [] #Suurimmat tasojen todennäköisyydet

#Etsitään rabi-taajuus ja AC Stark siirtymä kun ajavan kentän amplitudi on välillä [0.2*f_q,0.4*f_q] (hbar=1)
for i in range (35,51):
    A = qubit.frequency*(i/100)
    params=find_params_eigenbasis(qubit, A, evolution_eigenbasis,np.cbrt(A/3.495e+07))
    deltas.append(params[0])
    pulse_lengths.append(params[1])
    max_vals.append(params[2])
    
    f_d = qubit.frequency/3-deltas[-1]

    vals=sim_pulse(f_d,evolution_eigenbasis,dim,1/(2*pulse_lengths[-1]))

    print(A)
    print(vals[0])
    print(vals[1])

    g_vals=[arr[0] for arr in vals[2]]
    e_vals=[arr[1] for arr in vals[2]]
    f_vals=[arr[2] for arr in vals[2]]

    plt.figure()
    plt.plot(vals[3], g_vals)
    plt.plot(vals[3], e_vals)
    plt.plot(vals[3], f_vals)
    plt.xlabel("Pulssin pituus(ns)")
    plt.ylabel("P")
    plt.title("Tasojen suurimmat todennäköisyydet")
    plt.show()

In [ ]:
A_vals=(np.arange(21)*0.01+0.2)qubit.frequency

plt.figure()
plt.plot(A_vals, deltas, 'o')
plt.xlabel("Amplitudi (GHz)")
plt.ylabel("delta=f_q/3-f_d (GHz)")
plt.title("AC stark siirtymä")
plt.show()
print(deltas)

plt.figure()
plt.plot(A_vals, pulse_lengths, 'o')
plt.xlabel("Amplitudi (GHz)")
plt.ylabel("T_pulse (ns)")
plt.title("Pi-pulssin pituus")
plt.show()
print(rabis)

g_vals=[arr[0] for arr in max_vals]
e_vals=[arr[1] for arr in max_vals]
f_vals=[arr[2] for arr in max_vals]

plt.figure()
plt.plot(rabis, g_vals)
plt.plot(rabis, e_vals)
plt.plot(rabis, f_vals)
plt.xlabel("Omega_R (GHz)")
plt.ylabel("P")
plt.title("Tasojen suurimmat todennäköisyydet")
plt.show()